In [3]:
import duckdb

con = duckdb.connect("../olist.duckdb")

with open("../sql/01_build_schema.sql", "r", encoding="utf-8") as f:
    script = f.read()

con.execute(script)
print("schema built")

schema built


In [4]:
con.execute("""
SELECT table_name, estimated_size AS rows
FROM duckdb_tables()
WHERE table_name NOT LIKE 'raw_%'
ORDER BY table_name
""").df()

,table_name,rows
0,dim_customer,96096
1,dim_date,791
2,dim_product,32951
3,dim_seller,3095
4,fact_order_items,109827
5,fact_orders,96184
6,stg_order_agg,98666
7,stg_order_review,98673
8,stg_valid_orders,96184


In [5]:
def check(name, sql, expected=0):
    got = con.execute(sql).fetchone()[0]
    status = "PASS" if got == expected else "FAIL"
    print(f"{status}  {name:45} got={got}")
    return got == expected

results = []

results.append(check("fact_orders: order_id is unique",
    "SELECT COUNT(*) FROM (SELECT order_id FROM fact_orders GROUP BY 1 HAVING COUNT(*)>1)"))

results.append(check("dim_customer: customer_unique_id is unique",
    "SELECT COUNT(*) FROM (SELECT customer_unique_id FROM dim_customer GROUP BY 1 HAVING COUNT(*)>1)"))

results.append(check("fact_orders: no negative revenue",
    "SELECT COUNT(*) FROM fact_orders WHERE order_revenue <= 0"))

results.append(check("fact_orders: no delivery before purchase",
    "SELECT COUNT(*) FROM fact_orders WHERE order_delivered_customer_date < order_purchase_timestamp"))

results.append(check("fact_orders: all inside analysis window",
    "SELECT COUNT(*) FROM fact_orders WHERE purchase_date < DATE '2017-01-01' OR purchase_date >= DATE '2018-09-01'"))

results.append(check("fact_orders: every customer exists in dim_customer",
    """SELECT COUNT(*) FROM fact_orders f
       LEFT JOIN dim_customer d ON f.customer_unique_id = d.customer_unique_id
       WHERE d.customer_unique_id IS NULL"""))

results.append(check("fact_orders: days_late never negative",
    "SELECT COUNT(*) FROM fact_orders WHERE days_late < 0"))

results.append(check("fact_order_items: every order exists in fact_orders",
    """SELECT COUNT(*) FROM fact_order_items i
       LEFT JOIN fact_orders f ON i.order_id = f.order_id
       WHERE f.order_id IS NULL"""))

print("\nALL PASSED" if all(results) else "\nSOME CHECKS FAILED — fix before continuing")

PASS  fact_orders: order_id is unique               got=0
PASS  dim_customer: customer_unique_id is unique    got=0
PASS  fact_orders: no negative revenue              got=0
PASS  fact_orders: no delivery before purchase      got=0
PASS  fact_orders: all inside analysis window       got=0
PASS  fact_orders: every customer exists in dim_customer got=0
PASS  fact_orders: days_late never negative         got=0
PASS  fact_order_items: every order exists in fact_orders got=0

ALL PASSED


In [6]:
con.execute("""
SELECT
    (SELECT COUNT(*) FROM raw_orders)                          AS raw_orders,
    (SELECT COUNT(*) FROM fact_orders)                         AS valid_orders,
    (SELECT COUNT(*) FROM raw_orders) -
    (SELECT COUNT(*) FROM fact_orders)                         AS excluded,
    (SELECT COUNT(DISTINCT customer_unique_id) FROM fact_orders) AS customers,
    (SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE customer_order_seq = 2)
            / COUNT(DISTINCT customer_unique_id), 2) FROM fact_orders) AS repeat_rate_pct,
    (SELECT ROUND(100.0 * AVG(CASE WHEN is_late THEN 1 ELSE 0 END), 2) FROM fact_orders) AS late_pct
FROM (SELECT 1)
""").df()

,raw_orders,valid_orders,excluded,customers,repeat_rate_pct,late_pct
0,99441,96184,3257,93080,2.99,6.79
